# Tissue region identification

In [ ]:
import os
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import warnings 
warnings.filterwarnings('ignore')

from spin import spin

In [ ]:
base_path = '/home/jiahao/wanglab/Data/Analyzed/2024-12-02-Mingrui-SCZ/'
input_path = os.path.join(base_path, "expr", 'cell type classification')

out_path = os.path.join(base_path, 'tissue region identification')
if not os.path.exists(out_path):
    os.mkdir(out_path)
    
fig_path = os.path.join(base_path, 'figures')
if not os.path.exists(fig_path):
    os.mkdir(fig_path)

sc.settings.figdir = fig_path

In [ ]:
adata = sc.read_h5ad(os.path.join(input_path, '2025-01-09-all-sample-cell-typing-lv3-subcluster-aligned.h5ad'))
adata.var['max_counts'] = adata.layers['raw'].toarray().max(axis=0)
adata

## SPIN

### Each coronal section

In [ ]:
# PFC
adata_pfc = adata[adata.obs['coronal_position']=='PFC', adata.var.index[adata.var['highly_variable']]].copy()
adata_pfc = spin(adata_pfc, batch_key='sample', resolution=1.2)

# ST
adata_st = adata[adata.obs['coronal_position']=='ST', adata.var.index[adata.var['highly_variable']]].copy()
adata_st = spin(adata_st, batch_key='sample', resolution=1.2)

# HP
adata_hp = adata[adata.obs['coronal_position']=='HP', adata.var.index[adata.var['highly_variable']]].copy()
adata_hp = spin(adata_hp, batch_key='sample', resolution=1.2)

In [ ]:
# adata_pfc = adata[adata.obs['coronal_position']=='PFC'].copy()
# adata_st = adata[adata.obs['coronal_position']=='ST'].copy()
# adata_hp = adata[adata.obs['coronal_position']=='HP'].copy()

In [ ]:
adata_pfc = spin(
    adata_pfc,
    batch_key='sample',
    resolution=0.7
)

In [ ]:
from datetime import datetime
date = datetime.today().strftime('%Y-%m-%d')
adata_pfc.write_h5ad(f"{out_path}/{date}-pfc-spin-res1.h5ad")

In [ ]:
adata_st = spin(
    adata_st,
    batch_key='sample',
    resolution=0.7
)

In [ ]:
from datetime import datetime
date = datetime.today().strftime('%Y-%m-%d')
adata_st.write_h5ad(f"{out_path}/{date}-st-spin-res0.7.h5ad")

In [ ]:
adata_hp = spin(
    adata_hp,
    batch_key='sample',
    resolution=0.7
)

In [ ]:
from datetime import datetime
date = datetime.today().strftime('%Y-%m-%d')
adata_hp.write_h5ad(f"{out_path}/{date}-hp-spin-res0.7.h5ad")

## Region Label

### PFC

In [ ]:
# adata_pfc = sc.read_h5ad(os.path.join(out_path, '2025-01-09-pfc-spin-res0.7.h5ad'))
# adata_st = sc.read_h5ad(os.path.join(out_path, '2025-01-09-st-spin-res0.7.h5ad'))
# adata_hp = sc.read_h5ad(os.path.join(out_path, '2025-01-09-hp-spin-res0.7.h5ad'))

In [ ]:
adata_pfc.obs['region_label'] = adata_pfc.obs['region'].values

In [ ]:
transfer_dict_l1 = {}

# Level_1
level_1_list = [
    'CTX_A_9-[L5]', #0
    'CTX_A_11/12/14-[L6/ILA6]', #1
    'CTX_A_3/4-[L2/3]', #2
    'CTX_A_7-[mPFC5]', #3
    'CTX_A_8-[L4]', #4
    'CTX_A_1/2-[L1]/MNG_1', #5
    'FT_1', #6
    'CTX_B_5-[ILA2/3]', #7
    'CTX_A_16', #8
    'CTX_B_3', #9
    'CTX_A_9-[L5]', #10
    'CTX_A_1/2-[L1]/MNG_1', #11
    'CTX_A_8-[L4]', #12
    'CTX_A_11/12/14-[L6/ILA6]', #13
    'CTX_A_9-[L5]', #14
    'CTX_A_9-[L5]', #15
    'CTX_A_8-[L4]', #16
]

for i in sorted(adata_pfc.obs['region'].unique()):
    transfer_dict_l1[i] = level_1_list[int(i)]

In [ ]:
adata_pfc.obs = adata_pfc.obs.replace({'region_label': transfer_dict_l1})

In [ ]:
num_rows = 2
num_cols = 4
fig, axes = plt.subplots(num_rows, num_cols, figsize =(num_cols * 5, num_rows*6))

axes_flat = axes.flatten()
for i,sample in enumerate(adata_pfc.obs['sample_id'].cat.categories.tolist()):
    sc.pl.spatial(adata_pfc[adata_pfc.obs['sample_id']==sample],color='region_label', spot_size=200,legend_loc=None, show=False,title =f'{sample}',ax=axes_flat[i])

In [ ]:
adata_pfc.obs['region_label'] = adata_pfc.obs['region_label'].cat.reorder_categories(sorted(adata_pfc.obs['region_label'].cat.categories), ordered=True)

In [ ]:
sc.pl.embedding(adata_pfc, basis='X_umap_spin', color='region_label')

In [ ]:
from datetime import datetime
date = datetime.today().strftime('%Y-%m-%d')
adata_pfc.write_h5ad(f"{out_path}/{date}-pfc-rgn-label.h5ad")

### ST

In [ ]:
num_rows = 2
num_cols = 4
fig, axes = plt.subplots(num_rows, num_cols, figsize =(num_cols * 5, num_rows*6))

axes_flat = axes.flatten()
for i,sample in enumerate(adata_st.obs['sample_id'].cat.categories.tolist()):
    print(sample)
    sc.pl.spatial(adata_st[adata_st.obs['sample_id']==sample],color='region', spot_size=200,legend_loc=None, show=False,title =f'{sample}',ax=axes_flat[i])

In [ ]:
sc.tl.rank_genes_groups(adata_st,'region',mask_var='highly_variable', method='wilcoxon')

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata_st, n_genes=5, dendrogram=False, standard_scale='var')
plt.show()

In [ ]:
sc.pl.embedding(adata_st, basis='X_umap_spin', color='region')

In [ ]:
adata_st.obs['region_label'] = adata_st.obs['region'].values

In [ ]:
transfer_dict_l1 = {}

# Level_1
level_1_list = [
    'CNU_1', #0
    'CTX_A_11/12-[L6]', #1
    'CTX_A_8-[L4]', #2
    'CTX_A_3/4-[L2/3]', #3
    'CTX_A_9-[L5]', #4
    'CNU_2', #5
    'CTX_A_16', #6
    'FT_1', #7
    'CTX_A_1/2-[L1]/MNG_1', #8
    'CNU_4', #9
    'CTX_B', #10
    'VS_1', #11
    'CNU_1', #12
    'CNU_1', #13
    'CNU_1', #14
]

for i in sorted(adata_st.obs['region'].unique()):
    transfer_dict_l1[i] = level_1_list[int(i)]

In [ ]:
adata_st.obs = adata_st.obs.replace({'region_label': transfer_dict_l1})

In [ ]:
adata_st.obs['region_label'] = adata_st.obs['region_label'].cat.reorder_categories(sorted(adata_st.obs['region_label'].cat.categories), ordered=True)

In [ ]:
num_rows = 2
num_cols = 4
fig, axes = plt.subplots(num_rows, num_cols, figsize =(num_cols * 5, num_rows*6))

axes_flat = axes.flatten()
for i,sample in enumerate(adata_st.obs['sample_id'].cat.categories.tolist()):
    sc.pl.spatial(adata_st[adata_st.obs['sample_id']==sample],color='region_label', spot_size=200,legend_loc=None, show=False,title =f'{sample}',ax=axes_flat[i])

In [ ]:
sc.pl.embedding(adata_st, basis='X_umap_spin', color='region_label')

In [ ]:
from datetime import datetime
date = datetime.today().strftime('%Y-%m-%d')
adata_st.write_h5ad(f"{out_path}/{date}-st-rgn-label.h5ad")

### HP

In [ ]:
num_rows = 2
num_cols = 4
fig, axes = plt.subplots(num_rows, num_cols, figsize =(num_cols * 5, num_rows*6))

axes_flat = axes.flatten()
for i,sample in enumerate(adata_hp.obs['sample_id'].cat.categories.tolist()):
    sc.pl.spatial(adata_hp[adata_hp.obs['sample_id']==sample],color='region', spot_size=200,legend_loc=None, show=False,title =f'{sample}',ax=axes_flat[i])
    print(sample)

In [ ]:
sc.pl.embedding(adata, basis='X_umap_spin', color='region')

In [ ]:
# adata_hp.var['highly_variable'] = adata.var['highly_variable']
sc.tl.rank_genes_groups(adata_hp,'region',mask_var='highly_variable', method='wilcoxon')

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata_hp, n_genes=5, dendrogram=False, standard_scale='var')

In [ ]:
sc.pl.embedding(adata_hp, basis='X_umap_spin', color='region')

In [ ]:
adata_hp.obs['region_label'] = adata_hp.obs['region'].values

In [ ]:
transfer_dict_l1 = {}

# Level_1
level_1_list = [
    'TH_1', #0
    'FT_1', #1
    'TH_2', #2
    'MB_P_MY', #3
    'CTX_HIP_1', #4
    'CTX_A_1', #5
    'CTX_A_12/15', #6
    'CTX_HIP_2', #7
    'TH_3', #8
    'TH_4-[RT]', #9
    'CTX_HIP_3', #10
    'CTX_A_10/13', #11
    'VS_1', #12
    'VS_1', #13
    'TH_1', #14
    'FT_1', #15
    'TH_1', #16
    'TH_1', #17
    'MB_P_MY', #18
]

for i in sorted(adata_hp.obs['region'].unique()):
    transfer_dict_l1[i] = level_1_list[int(i)]


In [ ]:
adata_hp.obs = adata_hp.obs.replace({'region_label': transfer_dict_l1})
adata_hp.obs['region_label'] = adata_hp.obs['region_label'].cat.reorder_categories(sorted(adata_hp.obs['region_label'].cat.categories), ordered=True)

In [ ]:
num_rows = 2
num_cols = 4
fig, axes = plt.subplots(num_rows, num_cols, figsize =(num_cols * 5, num_rows*6))

axes_flat = axes.flatten()
for i,sample in enumerate(adata_hp.obs['sample_id'].cat.categories.tolist()):
    sc.pl.spatial(adata_hp[adata_hp.obs['sample_id']==sample],color='region_label', spot_size=200,legend_loc=None, show=False,title =f'{sample}',ax=axes_flat[i])


In [ ]:
sc.pl.embedding(adata_hp, basis='X_umap_spin', color='region_label')

In [ ]:
from datetime import datetime
date = datetime.today().strftime('%Y-%m-%d')
adata_hp.write_h5ad(f"{out_path}/{date}-hp-rgn-label.h5ad")